In [1]:
# [CELL 1] STUDENT CONFIGURATION, DATA LOADING, AND FEATURE SCREENING
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Pacudan"
STUDENT_ID = "TUPM-26-1859"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

# Find the SECOM data folder in this folder or any parent folder
WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR


def find_data_dir():
    for base in [WORK_DIR, *WORK_DIR.parents]:
        if (base / "data" / "secom.data").is_file():
            return base / "data"
    raise FileNotFoundError("data/secom.data not found in this folder or any parent folder.")

DATA_DIR = find_data_dir()
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw.ndim == 2 and len(labels) == len(raw)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
is_pass = labels == -1
is_fail = labels == 1

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Workspace:", WORK_DIR.resolve())
print("Data Folder:", DATA_DIR.resolve())
print("Output Folder:", OUTPUT_DIR.resolve())
print("Measurement Matrix Shape:", raw.shape)
print("Pass Records:", int(is_pass.sum()), "| Fail Records:", int(is_fail.sum()))
# Feature screening shared by the additional exercises
missing_mask = np.isnan(raw)
observed_n = (~missing_mask).sum(axis=0)
obs_min = np.where(missing_mask, np.inf, raw).min(axis=0)
obs_max = np.where(missing_mask, -np.inf, raw).max(axis=0)

has_missing = missing_mask.any(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = ~(all_missing | constant)
usable_idx = np.flatnonzero(usable)
print("Usable Features:", len(usable_idx))


Student: Pacudan TUPM-26-1859
Seed Digit: 9
ID Sum: 31
Name Length: 7
Random Seed: 261859
Verification Value: 226
Workspace: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Data Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\data
Output Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Measurement Matrix Shape: (1567, 590)
Pass Records: 1463 | Fail Records: 104
Usable Features: 474


In [2]:
# [CELL 2] EXERCISE 2 - GROUPED STATISTICS AND STANDARDIZED MEAN DIFFERENCE
# Requirement 1: labeled DataFrame preserving original SECOM record identifiers (the index)
df = pd.DataFrame(raw, columns=feature_names)
df.index.name = "Record_ID"
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_count = int((df["Outcome"] == "Pass").sum())
fail_count = int((df["Outcome"] == "Fail").sum())

# Requirement 2: reproducible balanced sample, min(20 + ID_SUM, Pass, Fail) per outcome
per_group = min(20 + ID_SUM, pass_count, fail_count)
rng2 = np.random.default_rng(RANDOM_SEED)

# Requirement 3: five usable features, median imputation using full-dataset medians
feat_pos = np.sort(rng2.choice(usable_idx, size=5, replace=False))
sel_names = feature_names[feat_pos].tolist()
full_medians = df[sel_names].median()

chosen = []
for outcome in ["Pass", "Fail"]:
    ids = df.index[df["Outcome"] == outcome].to_numpy()
    chosen.append(rng2.choice(ids, size=per_group, replace=False))
chosen_ids = np.sort(np.concatenate(chosen))

sample = df.loc[chosen_ids, sel_names + ["Result", "Outcome"]].copy()
missing_in_sample = int(sample[sel_names].isna().sum().sum())
sample[sel_names] = sample[sel_names].fillna(full_medians)

# Requirement 4: groupby + agg (count, mean, median, sample SD)
stats = sample.groupby("Outcome")[sel_names].agg(["count", "mean", "median", "std"])

# Requirement 5: absolute and standardized mean difference (pooled within-group SD)
rows = []
for f in sel_names:
    p, fl = stats.loc["Pass", f], stats.loc["Fail", f]
    n_p, n_f = p["count"], fl["count"]
    pooled = np.sqrt(((n_p - 1) * p["std"] ** 2 + (n_f - 1) * fl["std"] ** 2) / (n_p + n_f - 2))
    abs_diff = abs(fl["mean"] - p["mean"])
    std_diff = abs_diff / pooled if pooled > 0 else np.nan   # zero denominator handled safely
    rows.append({
        "Feature": f,
        "Pass_Count": int(n_p), "Fail_Count": int(n_f),
        "Pass_Mean": p["mean"], "Fail_Mean": fl["mean"],
        "Pass_Median": p["median"], "Fail_Median": fl["median"],
        "Pass_SD": p["std"], "Fail_SD": fl["std"],
        "Abs_Mean_Diff": abs_diff, "Pooled_SD": pooled,
        "Abs_Std_Mean_Diff": std_diff,
    })

# Requirement 6: rank by absolute standardized mean difference and export
comparison = (pd.DataFrame(rows)
              .sort_values("Abs_Std_Mean_Diff", ascending=False, na_position="last")
              .reset_index(drop=True))
comparison.insert(0, "Rank", comparison.index + 1)

sample_path = OUTPUT_DIR / "balanced_sample.csv"
comparison_path = OUTPUT_DIR / "feature_comparison.csv"
sample.reset_index().to_csv(sample_path, index=False)
comparison.to_csv(comparison_path, index=False)

# Verification: NumPy recomputation must match the pandas groupby results
np_ok = True
for f in sel_names:
    for outcome in ["Pass", "Fail"]:
        v = sample.loc[sample["Outcome"] == outcome, f].to_numpy()
        np_ok &= bool(np.isclose(v.mean(), stats.loc[outcome, (f, "mean")]))
        np_ok &= bool(np.isclose(np.median(v), stats.loc[outcome, (f, "median")]))
        np_ok &= bool(np.isclose(v.std(ddof=1), stats.loc[outcome, (f, "std")]))
        np_ok &= int(v.size) == int(stats.loc[outcome, (f, "count")])

reloaded = pd.read_csv(sample_path)
export_ok = len(reloaded) == 2 * per_group and np.array_equal(reloaded["Record_ID"], sample.index.to_numpy())
top = comparison.iloc[0]

print("Pass Records:", pass_count, "| Fail Records:", fail_count)
print("Records Per Outcome:", per_group)
print("Missing Values Imputed In Sample:", missing_in_sample)
print("Selected Features:", sel_names)
print()
print(comparison.round(4).to_string(index=False))
print()
print("=" * 70)
print("EXERCISE 2 - ASSESSMENT DATA")
print("=" * 70)
print(f"{'Balanced Sample Size':<34}: {per_group} per outcome ({len(sample)} total)")
print(f"{'Selected Features':<34}: {', '.join(sel_names)}")
print(f"{'Highest-Ranked Feature':<34}: {top['Feature']}")
print(f"{'Absolute Standardized Mean Diff.':<34}: {top['Abs_Std_Mean_Diff']:.4f}")
print(f"{'NumPy-Pandas Statistics Match':<34}: {np_ok}")
print(f"{'Required Outputs':<34}: {sample_path.name}, {comparison_path.name} (exports verified: {export_ok})")
print("=" * 70)

Pass Records: 1463 | Fail Records: 104
Records Per Outcome: 51
Missing Values Imputed In Sample: 54
Selected Features: ['Feature_154', 'Feature_385', 'Feature_467', 'Feature_471', 'Feature_567']

 Rank     Feature  Pass_Count  Fail_Count  Pass_Mean  Fail_Mean  Pass_Median  Fail_Median  Pass_SD  Fail_SD  Abs_Mean_Diff  Pooled_SD  Abs_Std_Mean_Diff
    1 Feature_385          51          51     0.0102     0.0134       0.0068       0.0069   0.0099   0.0112         0.0032     0.0105             0.3017
    2 Feature_471          51          51     9.7069     8.8109       8.4927       7.5116   6.0578   5.1786         0.8960     5.6354             0.1590
    3 Feature_567          51          51     0.0611     0.0675       0.0479       0.0486   0.0398   0.0425         0.0064     0.0412             0.1557
    4 Feature_467          51          51     6.2219     7.5444       5.7506       5.9363   3.4338  13.7449         1.3225    10.0178             0.1320
    5 Feature_154          51          